<a href="https://colab.research.google.com/github/P4bl1t0ck/ProyectoFinal_Analisis-VisualizacionDeDatos/blob/master/ETL_RetailMax_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ETL RetailMax S.A. — Fase 2: Ingeniería de Datos y DataMart

Este notebook extrae las 3 fuentes sucias (`transacciones_sucio.csv`, `productos_sucio.csv`, `vendedores_sucio.csv`), limpia y valida los datos, construye el modelo en estrella (5 dimensiones + 1 hecho) generando las claves subrogadas en Python, y exporta 6 CSV listos para cargar en SQL Server usando `datamart_ddl.sql`.

**Antes de correr:** sube los 3 CSV sucios a la sesión de Colab (panel izquierdo → ícono de carpeta → subir archivo), o monta tu Google Drive.

## 0. Subir los archivos de origen a Colab

In [ ]:
# Ejecuta esta celda para subir manualmente los 3 CSV sucios desde tu computadora
from google.colab import files

uploaded = files.upload()  # selecciona transacciones_sucio.csv, productos_sucio.csv y vendedores_sucio.csv

Saving productos_sucio.csv to productos_sucio (1).csv
Saving transacciones_sucio.csv to transacciones_sucio (1).csv
Saving vendedores_sucio.csv to vendedores_sucio (1).csv


Alternativa: si prefieres montar Google Drive en vez de subir manualmente cada vez, descomenta y ajusta la ruta:
```python
from google.colab import drive
drive.mount('/content/drive')
# Luego ajusta las rutas de lectura en la Etapa 1 a tu carpeta de Drive
```

In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

pd.set_option("display.max_columns", None)

REPORTE = []  # acumula mensajes del reporte de calidad de datos

def log(msg):
    print(msg)
    REPORTE.append(msg)

## 1. Extract — Carga de las 3 fuentes

In [ ]:
log("=" * 70)
log("ETAPA 1: EXTRACT")
log("=" * 70)

df_t_raw = pd.read_csv("transacciones_sucio.csv")
df_p_raw = pd.read_csv("productos_sucio.csv")
df_v_raw = pd.read_csv("vendedores_sucio.csv")

log(f"transacciones_sucio.csv : {len(df_t_raw)} filas")
log(f"productos_sucio.csv     : {len(df_p_raw)} filas")
log(f"vendedores_sucio.csv    : {len(df_v_raw)} filas")

ETAPA 1: EXTRACT
transacciones_sucio.csv : 30315 filas
productos_sucio.csv     : 3030 filas
vendedores_sucio.csv    : 91 filas


## 2. Transform — Limpieza de cada fuente

In [ ]:
log("\n" + "=" * 70)
log("ETAPA 2: TRANSFORM - LIMPIEZA")
log("=" * 70)


ETAPA 2: TRANSFORM - LIMPIEZA


### 2.1 Limpieza de PRODUCTOS

In [ ]:
log("\n--- Limpieza de productos ---")
dfp = df_p_raw.copy()
n_inicial = len(dfp)

# Quitar espacios en codigo_producto y texto en general
dfp["codigo_producto"] = dfp["codigo_producto"].astype(str).str.strip()
for col in ["nombre_producto", "descripcion", "subcategoria", "marca", "proveedor"]:
    dfp[col] = dfp[col].astype(str).str.strip().replace({"nan": np.nan})

# Normalizar categoria: quitar espacios, tildes, mayusculas -> Title Case estandar
def normalizar_categoria(c):
    if pd.isna(c):
        return c
    c = str(c).strip()
    c = c.replace("á", "a").replace("é", "e").replace("í", "i").replace("ó", "o").replace("ú", "u")
    c = c.lower()
    mapping = {
        "electronica": "Electronica",
        "hogar y decoracion": "Hogar y Decoracion",
        "ropa y accesorios": "Ropa y Accesorios",
        "alimentos y bebidas": "Alimentos y Bebidas",
    }
    return mapping.get(c, c.title())

dfp["categoria"] = dfp["categoria"].apply(normalizar_categoria)

# precio_lista: quitar simbolo $ y convertir a numerico
dfp["precio_lista"] = (
    dfp["precio_lista"].astype(str).str.replace("$", "", regex=False).str.strip()
)
dfp["precio_lista"] = pd.to_numeric(dfp["precio_lista"], errors="coerce")
dfp["costo_estandar"] = pd.to_numeric(dfp["costo_estandar"], errors="coerce")

# Eliminar duplicados de codigo_producto (se conserva el primero)
dup_p = dfp.duplicated(subset=["codigo_producto"]).sum()
dfp = dfp.drop_duplicates(subset=["codigo_producto"], keep="first")
log(f"Duplicados de codigo_producto eliminados: {dup_p}")

# Eliminar filas sin codigo_producto o sin categoria (no se pueden ubicar)
antes = len(dfp)
dfp = dfp.dropna(subset=["codigo_producto", "categoria", "precio_lista"])
log(f"Filas descartadas por codigo/categoria/precio nulo: {antes - len(dfp)}")

# Imputar costo_estandar nulo con un porcentaje promedio del precio por categoria
costo_pct_promedio = (dfp["costo_estandar"] / dfp["precio_lista"]).median()
mask_costo_nulo = dfp["costo_estandar"].isna()
dfp.loc[mask_costo_nulo, "costo_estandar"] = (dfp.loc[mask_costo_nulo, "precio_lista"] * costo_pct_promedio).round(2)
log(f"Costos nulos imputados (pct mediano sobre precio): {mask_costo_nulo.sum()}")

# Corregir registros donde precio_lista < costo_estandar (margen negativo, error de captura)
mask_margen_neg = dfp["precio_lista"] < dfp["costo_estandar"]
log(f"Productos con precio < costo (margen negativo) corregidos: {mask_margen_neg.sum()}")
dfp.loc[mask_margen_neg, "precio_lista"] = (dfp.loc[mask_margen_neg, "costo_estandar"] * 1.3).round(2)

# Imputar marca/proveedor/subcategoria nulos con "Sin especificar"
for col in ["marca", "proveedor", "subcategoria"]:
    n_nulos = dfp[col].isna().sum()
    dfp[col] = dfp[col].fillna("Sin especificar")
    log(f"Nulos imputados en {col}: {n_nulos}")

log(f"productos: {n_inicial} -> {len(dfp)} filas limpias")


--- Limpieza de productos ---
Duplicados de codigo_producto eliminados: 30
Filas descartadas por codigo/categoria/precio nulo: 0
Costos nulos imputados (pct mediano sobre precio): 30
Productos con precio < costo (margen negativo) corregidos: 15
Nulos imputados en marca: 60
Nulos imputados en proveedor: 44
Nulos imputados en subcategoria: 30
productos: 3030 -> 3000 filas limpias


### 2.2 Limpieza de VENDEDORES

In [ ]:
log("\n--- Limpieza de vendedores ---")
dfv = df_v_raw.copy()
n_inicial = len(dfv)

dfv["id_vendedor"] = dfv["id_vendedor"].astype(str).str.strip()
dfv["nombre_completo"] = dfv["nombre_completo"].astype(str).str.strip()
dfv["nombre_completo"] = dfv["nombre_completo"].str.replace(r"\s+", " ", regex=True).str.title()
dfv["supervisor"] = dfv["supervisor"].astype(str).str.strip().replace({"nan": np.nan})
dfv["ciudad"] = dfv["ciudad"].astype(str).str.strip().replace({"nan": np.nan})
dfv["tienda"] = dfv["tienda"].astype(str).str.strip()

# Normalizar region (typos, mayusculas, espacios)
def normalizar_region(r):
    if pd.isna(r):
        return r
    r = str(r).strip().lower()
    mapping = {
        "sierra": "Sierra", "sierrra": "Sierra",
        "costa": "Costa", "csota": "Costa",
        "austro": "Austro",
    }
    return mapping.get(r, r.title())

dfv["region"] = dfv["region"].apply(normalizar_region)

# Normalizar estado a 'Activo' / 'Inactivo'
def normalizar_estado(e):
    if pd.isna(e):
        return np.nan
    e = str(e).strip().lower()
    if e in ["activo", "active", "1", "si"]:
        return "Activo"
    if e in ["inactivo", "inactive", "0", "no"]:
        return "Inactivo"
    return e.title()

dfv["estado"] = dfv["estado"].apply(normalizar_estado)

# Fecha de ingreso: parseo flexible, descartar fechas futuras (imposibles)
dfv["fecha_ingreso"] = pd.to_datetime(dfv["fecha_ingreso"], errors="coerce")
hoy = pd.Timestamp(datetime.now().date())
mask_futura = dfv["fecha_ingreso"] > hoy
log(f"Fechas de ingreso futuras (invalidas) detectadas: {mask_futura.sum()}")
dfv.loc[mask_futura, "fecha_ingreso"] = pd.NaT

# Eliminar duplicados de id_vendedor
dup_v = dfv.duplicated(subset=["id_vendedor"]).sum()
dfv = dfv.drop_duplicates(subset=["id_vendedor"], keep="first")
log(f"Duplicados de id_vendedor eliminados: {dup_v}")

# Descartar filas sin id_vendedor, tienda o region (no ubicables)
antes = len(dfv)
dfv = dfv.dropna(subset=["id_vendedor", "tienda", "region", "estado"])
log(f"Filas descartadas por campos clave nulos: {antes - len(dfv)}")

# Imputar supervisor y fecha_ingreso nulos
n_sup_nulo = dfv["supervisor"].isna().sum()
dfv["supervisor"] = dfv["supervisor"].fillna("Sin asignar")
log(f"Nulos imputados en supervisor: {n_sup_nulo}")

n_fecha_nula = dfv["fecha_ingreso"].isna().sum()
dfv["fecha_ingreso"] = dfv["fecha_ingreso"].fillna(pd.Timestamp("2020-01-01"))
log(f"Nulos/invalidas imputados en fecha_ingreso (valor por defecto): {n_fecha_nula}")

log(f"vendedores: {n_inicial} -> {len(dfv)} filas limpias")


--- Limpieza de vendedores ---
Fechas de ingreso futuras (invalidas) detectadas: 1
Duplicados de id_vendedor eliminados: 3
Filas descartadas por campos clave nulos: 0
Nulos imputados en supervisor: 5
Nulos/invalidas imputados en fecha_ingreso (valor por defecto): 4
vendedores: 91 -> 88 filas limpias


### 2.3 Limpieza de TRANSACCIONES

In [ ]:
log("\n--- Limpieza de transacciones ---")
dft = df_t_raw.copy()
n_inicial = len(dft)

# Eliminar filas completamente vacias (basura)
antes = len(dft)
dft = dft.dropna(how="all")
log(f"Filas completamente vacias eliminadas: {antes - len(dft)}")

# Eliminar duplicados exactos de id_transaccion + codigo_producto
antes = len(dft)
dft = dft.drop_duplicates(subset=["id_transaccion", "codigo_producto"], keep="first")
log(f"Duplicados de transaccion eliminados: {antes - len(dft)}")

# Normalizar id_vendedor (minusculas, sin cero relleno, espacios)
def normalizar_id_vendedor(v):
    if pd.isna(v):
        return v
    v = str(v).strip().upper()
    m = re.match(r"V0*(\d+)", v)
    if m:
        return f"V{int(m.group(1)):04d}"
    return v

dft["id_vendedor"] = dft["id_vendedor"].apply(normalizar_id_vendedor)

# Normalizar tienda (espacios, mayusculas/minusculas)
dft["tienda"] = dft["tienda"].astype(str).str.strip()
dft["tienda"] = dft["tienda"].str.replace(r"\s+", " ", regex=True)
dft["tienda"] = dft["tienda"].replace({"nan": np.nan})
# Convertir a Title Case estandar para que coincida con DimTienda
dft["tienda"] = dft["tienda"].apply(lambda x: x.title() if pd.notna(x) else x)

# Parseo flexible de fecha_transaccion (multiples formatos)
def parsear_fecha(f):
    if pd.isna(f):
        return pd.NaT
    f = str(f).strip()
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y", "%d.%m.%Y"):
        try:
            return pd.to_datetime(f, format=fmt)
        except ValueError:
            continue
    return pd.NaT

dft["fecha_transaccion"] = dft["fecha_transaccion"].apply(parsear_fecha)
n_fecha_invalida = dft["fecha_transaccion"].isna().sum()
log(f"Fechas no parseables: {n_fecha_invalida}")

# precio_unitario_lista: convertir coma decimal a punto y castear a numerico
dft["precio_unitario_lista"] = (
    dft["precio_unitario_lista"].astype(str).str.replace(",", ".", regex=False)
)
dft["precio_unitario_lista"] = pd.to_numeric(dft["precio_unitario_lista"], errors="coerce")

for col in ["cantidad", "pct_descuento", "venta_bruta", "monto_descuento", "venta_neta", "costo_total"]:
    dft[col] = pd.to_numeric(dft[col], errors="coerce")

# Reglas de negocio (seccion 3.8 del documento):
# - pct_descuento debe ser >= 0
n_desc_neg = (dft["pct_descuento"] < 0).sum()
dft.loc[dft["pct_descuento"] < 0, "pct_descuento"] = dft["pct_descuento"].abs()
log(f"Descuentos negativos corregidos (valor absoluto): {n_desc_neg}")

# - pct_descuento no puede superar 100%
n_desc_alto = (dft["pct_descuento"] > 100).sum()
dft = dft[~(dft["pct_descuento"] > 100)]
log(f"Filas con descuento > 100% descartadas (dato no confiable): {n_desc_alto}")

# - cantidad razonable: outliers de digitacion (> 50 unidades en una sola linea)
n_cant_outlier = (dft["cantidad"] > 50).sum()
dft = dft[~(dft["cantidad"] > 50)]
log(f"Filas con cantidad > 50 descartadas (outlier de digitacion): {n_cant_outlier}")

# Recalcular venta_bruta, monto_descuento y venta_neta de forma consistente
# (en vez de confiar en columnas potencialmente inconsistentes)
dft["venta_bruta"] = (dft["precio_unitario_lista"] * dft["cantidad"]).round(2)
dft["monto_descuento"] = (dft["venta_bruta"] * dft["pct_descuento"] / 100).round(2)
dft["venta_neta"] = (dft["venta_bruta"] - dft["monto_descuento"]).round(2)
log("venta_bruta, monto_descuento y venta_neta recalculadas para garantizar consistencia")

# Eliminar filas con campos clave nulos despues de la limpieza
antes = len(dft)
dft = dft.dropna(subset=[
    "id_transaccion", "fecha_transaccion", "id_vendedor", "tienda",
    "codigo_producto", "cantidad", "precio_unitario_lista", "pct_descuento", "costo_total"
])
log(f"Filas descartadas por campos clave nulos tras limpieza: {antes - len(dft)}")

# Eliminar transacciones huerfanas: producto o vendedor que no existe en las dimensiones limpias
antes = len(dft)
dft = dft[dft["codigo_producto"].isin(dfp["codigo_producto"])]
log(f"Transacciones con producto inexistente descartadas: {antes - len(dft)}")

antes = len(dft)
dft = dft[dft["id_vendedor"].isin(dfv["id_vendedor"])]
log(f"Transacciones con vendedor inexistente descartadas: {antes - len(dft)}")

log(f"transacciones: {n_inicial} -> {len(dft)} filas limpias")


--- Limpieza de transacciones ---
Filas completamente vacias eliminadas: 15
Duplicados de transaccion eliminados: 294
Fechas no parseables: 0
Descuentos negativos corregidos (valor absoluto): 141
Filas con descuento > 100% descartadas (dato no confiable): 87
Filas con cantidad > 50 descartadas (outlier de digitacion): 89
venta_bruta, monto_descuento y venta_neta recalculadas para garantizar consistencia
Filas descartadas por campos clave nulos tras limpieza: 447
Transacciones con producto inexistente descartadas: 118
Transacciones con vendedor inexistente descartadas: 0
transacciones: 30315 -> 29265 filas limpias


## 3. Transform — Construcción del modelo en estrella

In [ ]:
log("\n" + "=" * 70)
log("ETAPA 3: TRANSFORM - MODELO EN ESTRELLA")
log("=" * 70)


ETAPA 3: TRANSFORM - MODELO EN ESTRELLA


### 3.1 DimTiempo (generada a partir del rango de fechas usadas)

In [ ]:
fecha_min = dft["fecha_transaccion"].min()
fecha_max = dft["fecha_transaccion"].max()
rango_fechas = pd.date_range(fecha_min, fecha_max, freq="D")

dias_es = {0: "Lunes", 1: "Martes", 2: "Miercoles", 3: "Jueves", 4: "Viernes", 5: "Sabado", 6: "Domingo"}
meses_es = {1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril", 5: "Mayo", 6: "Junio",
            7: "Julio", 8: "Agosto", 9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"}

dim_tiempo = pd.DataFrame({"fecha": rango_fechas})
dim_tiempo["id_tiempo_sk"] = dim_tiempo["fecha"].dt.strftime("%Y%m%d").astype(int)
dim_tiempo["dia"] = dim_tiempo["fecha"].dt.day
dim_tiempo["mes"] = dim_tiempo["fecha"].dt.month
dim_tiempo["nombre_mes"] = dim_tiempo["mes"].map(meses_es)
dim_tiempo["trimestre"] = dim_tiempo["fecha"].dt.quarter
dim_tiempo["anio"] = dim_tiempo["fecha"].dt.year
dim_tiempo["dia_semana"] = dim_tiempo["fecha"].dt.weekday + 1
dim_tiempo["nombre_dia"] = (dim_tiempo["dia_semana"] - 1).map(dias_es)
dim_tiempo["es_fin_semana"] = dim_tiempo["dia_semana"].isin([6, 7]).astype(int)
dim_tiempo["semestre"] = np.where(dim_tiempo["mes"] <= 6, 1, 2)
dim_tiempo = dim_tiempo[[
    "id_tiempo_sk", "fecha", "dia", "mes", "nombre_mes", "trimestre",
    "anio", "dia_semana", "nombre_dia", "es_fin_semana", "semestre"
]]
log(f"DimTiempo generada: {len(dim_tiempo)} fechas ({fecha_min.date()} a {fecha_max.date()})")

DimTiempo generada: 365 fechas (2025-01-01 a 2025-12-31)


### 3.2 DimRegion

In [ ]:
regiones_unicas = sorted(dfv["region"].dropna().unique())
dim_region = pd.DataFrame({
    "id_region_sk": range(1, len(regiones_unicas) + 1),
    "nombre_region": regiones_unicas,
})
log(f"DimRegion generada: {len(dim_region)} regiones -> {regiones_unicas}")

DimRegion generada: 3 regiones -> ['Austro', 'Costa', 'Sierra']


### 3.3 DimTienda (tienda + ciudad, vinculada a region)

In [ ]:
tiendas_unicas = dfv[["tienda", "ciudad", "region"]].dropna().drop_duplicates(subset=["tienda"])
tiendas_unicas = tiendas_unicas.merge(dim_region, left_on="region", right_on="nombre_region", how="left")
dim_tienda = tiendas_unicas[["tienda", "ciudad", "id_region_sk"]].reset_index(drop=True)
dim_tienda.insert(0, "id_tienda_sk", range(1, len(dim_tienda) + 1))
dim_tienda = dim_tienda.rename(columns={"tienda": "nombre_tienda"})
log(f"DimTienda generada: {len(dim_tienda)} tiendas")

DimTienda generada: 11 tiendas


### 3.4 DimProducto

In [ ]:
dim_producto = dfp[[
    "codigo_producto", "nombre_producto", "descripcion", "categoria",
    "subcategoria", "marca", "proveedor", "costo_estandar", "precio_lista"
]].reset_index(drop=True)
dim_producto.insert(0, "id_producto_sk", range(1, len(dim_producto) + 1))
log(f"DimProducto generada: {len(dim_producto)} productos")

# --------------------------------------------------------------
# 3.5 DimVendedor (SCD Tipo 2)
# Como el dataset fuente no trae historial real de cambios, se genera
# la version 1 (vigente) de cada vendedor con fecha_inicio = fecha_ingreso.
# El campo version/es_actual queda listo para que futuras cargas
# incrementales puedan cerrar esta version y abrir una nueva si cambia
# tienda, region, supervisor o estado.
# --------------------------------------------------------------
dfv_merge = dfv.merge(dim_tienda, left_on="tienda", right_on="nombre_tienda", how="left")
# dim_tienda ya trae id_region_sk (resuelto en el paso 3.3), no se requiere merge adicional

# Descartar vendedores que no pudieron ubicarse en una tienda valida
antes = len(dfv_merge)
dfv_merge = dfv_merge.dropna(subset=["id_tienda_sk", "id_region_sk"])
log(f"Vendedores descartados por tienda/region no ubicable: {antes - len(dfv_merge)}")

dim_vendedor = dfv_merge[[
    "id_vendedor", "nombre_completo", "supervisor", "fecha_ingreso",
    "estado", "id_tienda_sk", "id_region_sk"
]].reset_index(drop=True)
dim_vendedor.insert(0, "id_vendedor_sk", range(1, len(dim_vendedor) + 1))
dim_vendedor["fecha_inicio_vigencia"] = dim_vendedor["fecha_ingreso"]
dim_vendedor["fecha_fin_vigencia"] = pd.NaT
dim_vendedor["es_actual"] = 1
dim_vendedor["version"] = 1
log(f"DimVendedor generada: {len(dim_vendedor)} vendedores (todos version 1, vigente)")

DimProducto generada: 3000 productos
Vendedores descartados por tienda/region no ubicable: 0
DimVendedor generada: 88 vendedores (todos version 1, vigente)


### 3.6 FactVentas (resolucion de claves subrogadas via lookup)

In [ ]:
fact = dft.copy()
fact["id_tiempo_sk"] = fact["fecha_transaccion"].dt.strftime("%Y%m%d").astype(int)

fact = fact.merge(
    dim_producto[["codigo_producto", "id_producto_sk"]],
    on="codigo_producto", how="inner"
)
fact = fact.merge(
    dim_vendedor[dim_vendedor["es_actual"] == 1][["id_vendedor", "id_vendedor_sk", "id_tienda_sk"]],
    on="id_vendedor", how="inner"
)

fact_ventas = fact[[
    "id_transaccion", "id_tiempo_sk", "id_producto_sk", "id_vendedor_sk", "id_tienda_sk",
    "cantidad", "precio_unitario_lista", "pct_descuento",
    "venta_bruta", "monto_descuento", "venta_neta", "costo_total"
]].reset_index(drop=True)
fact_ventas.insert(0, "id_venta_sk", range(1, len(fact_ventas) + 1))
log(f"FactVentas generada: {len(fact_ventas)} filas (con todas las FK resueltas)")

FactVentas generada: 29265 filas (con todas las FK resueltas)


## 4. Load — Exportación de los 6 CSV finales

In [ ]:
log("\n" + "=" * 70)
log("ETAPA 4: LOAD - EXPORTACION")
log("=" * 70)

dim_tiempo.to_csv("dim_tiempo.csv", index=False, encoding="utf-8")
dim_region.to_csv("dim_region.csv", index=False, encoding="utf-8")
dim_tienda.to_csv("dim_tienda.csv", index=False, encoding="utf-8")
dim_producto.to_csv("dim_producto.csv", index=False, encoding="utf-8")
dim_vendedor.to_csv("dim_vendedor.csv", index=False, encoding="utf-8")
fact_ventas.to_csv("fact_ventas.csv", index=False, encoding="utf-8")

log("Archivos generados: dim_tiempo.csv, dim_region.csv, dim_tienda.csv, "
    "dim_producto.csv, dim_vendedor.csv, fact_ventas.csv")


ETAPA 4: LOAD - EXPORTACION
Archivos generados: dim_tiempo.csv, dim_region.csv, dim_tienda.csv, dim_producto.csv, dim_vendedor.csv, fact_ventas.csv


## 5. Reporte de calidad de datos

In [ ]:
with open("reporte_calidad_datos.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(REPORTE))
log("\nReporte de calidad de datos guardado en reporte_calidad_datos.txt")

log("\nETL finalizado correctamente.")


Reporte de calidad de datos guardado en reporte_calidad_datos.txt

ETL finalizado correctamente.


## 6. Vista previa de los resultados

Verificación rápida de integridad referencial antes de descargar los archivos.

In [ ]:
print("Filas por tabla:")
print(f"  dim_tiempo    : {len(dim_tiempo)}")
print(f"  dim_region    : {len(dim_region)}")
print(f"  dim_tienda    : {len(dim_tienda)}")
print(f"  dim_producto  : {len(dim_producto)}")
print(f"  dim_vendedor  : {len(dim_vendedor)}")
print(f"  fact_ventas   : {len(fact_ventas)}")

print("\nValidacion de integridad referencial (debe ser 0 en todos los casos):")
print("  FK tiempo huerfanas  :", (~fact_ventas.id_tiempo_sk.isin(dim_tiempo.id_tiempo_sk)).sum())
print("  FK producto huerfanas:", (~fact_ventas.id_producto_sk.isin(dim_producto.id_producto_sk)).sum())
print("  FK vendedor huerfanas:", (~fact_ventas.id_vendedor_sk.isin(dim_vendedor.id_vendedor_sk)).sum())
print("  FK tienda huerfanas  :", (~fact_ventas.id_tienda_sk.isin(dim_tienda.id_tienda_sk)).sum())
print("  Descuentos fuera de rango [0,100]:", ((fact_ventas.pct_descuento < 0) | (fact_ventas.pct_descuento > 100)).sum())

fact_ventas.head()

Filas por tabla:
  dim_tiempo    : 365
  dim_region    : 3
  dim_tienda    : 11
  dim_producto  : 3000
  dim_vendedor  : 88
  fact_ventas   : 29265

Validacion de integridad referencial (debe ser 0 en todos los casos):
  FK tiempo huerfanas  : 0
  FK producto huerfanas: 0
  FK vendedor huerfanas: 0
  FK tienda huerfanas  : 0
  Descuentos fuera de rango [0,100]: 0


,id_venta_sk,id_transaccion,id_tiempo_sk,id_producto_sk,id_vendedor_sk,id_tienda_sk,cantidad,precio_unitario_lista,pct_descuento,venta_bruta,monto_descuento,venta_neta,costo_total
0,1,T0000173,20250427,1170,68,2,1.0,39.46,13.76,39.46,5.43,34.03,24.44
1,2,T0006809,20251004,125,68,2,1.0,16.14,2.93,16.14,0.47,15.67,13.01
2,3,T0002275,20251107,1188,58,7,2.0,90.54,29.61,181.08,53.62,127.46,68.81
3,4,T0012431,20250413,2079,12,8,2.0,164.71,6.57,329.42,21.64,307.78,138.76
4,5,T0004964,20250503,2509,61,8,1.0,135.54,17.81,135.54,24.14,111.40,50.65


## 7. Descargar los 6 CSV finales

In [ ]:
from google.colab import files

for nombre in ["dim_tiempo.csv", "dim_region.csv", "dim_tienda.csv",
               "dim_producto.csv", "dim_vendedor.csv", "fact_ventas.csv",
               "reporte_calidad_datos.txt"]:
    files.download(nombre)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>